# PartnerCopilot V3: rodada final congelada
Novo notebook e kernel novo. GPU, Internet e HF_TOKEN. Nao anexar resultados antigos.
Executar UMA rodada: primeiro n11-n20 (3 negativas), depois as 30 antigas marcadas
contaminadas, no mesmo processo/modelo. Nao olhar perguntas novas para ajustar o
prompt nem executar smoke test nelas. Nao mudar o protocolo apos ver as respostas.
A prova literal de trecho nao garante fidelidade ou relevancia: a leitura e separada.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import torch
assert torch.cuda.is_available(), 'Ative GPU.'
assert torch.cuda.device_count() == 1, 'Reinicie o kernel.'
print(torch.cuda.get_device_name(0))
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Habilite HF_TOKEN neste notebook.'

## Codigo congelado
Clone separado. Nao faz pull automatico ao retomar. Codigo e dados precisam estar
publicados no GitHub. O processo de inferencia inicia depois da instalacao.

In [ ]:
repo = Path('/kaggle/working/partner-copilot-final-v3')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/JoaoNordemann/partner-copilot.git', str(repo)], check=True)
os.chdir(repo)
assert Path('scripts/evaluate_final_v3.py').is_file(), 'Publique o codigo V3 antes de executar.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Verificar sem inferencia
Confere hashes do prompt, parser e snapshots. Mostra contagens, nao perguntas.
Esperado: 10/3 novas, 30/6 contaminadas, contexto 8, max_new_tokens 512, retry false.

In [ ]:
output_dir = Path('/kaggle/working/PartnerCopilot/final-v3')
command = [sys.executable, 'scripts/evaluate_final_v3.py', '--output-dir', str(output_dir), '--suite', 'both']
subprocess.run(command + ['--check-only'], check=True)

## Executar uma vez
O script processa as novas primeiro, depois o registro contaminado. Nao modificar
instrucoes ou configuracao entre os grupos. --resume pula registros ja completos.
Nenhuma nova tentativa corretiva e feita em erro de formato, ID ou trecho.

In [ ]:
subprocess.run(command + ['--resume'], check=True)

## Resumos e leitura
Nao misturar os dois grupos. Schema valido, ID valido e trecho literal nao sao
factualidade. Preencha a revisao depois de ler cada resposta e sua fonte.

In [ ]:
subprocess.run(command + ['--summarize-only'], check=True)
for cohort in ('new', 'contaminated'):
    print('GRUPO:', cohort)
    print((output_dir / f'generation-v3-{cohort}.summary.json').read_text(encoding='utf-8'))

## Preservar
Baixe os seis arquivos (dois JSONLs, dois summaries, duas reviews) antes de encerrar.
Os templates de revisao ja existentes nao sao sobrescritos pela recalculacao.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
zip_path = Path('/kaggle/working/partner-copilot-final-v3.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in output_dir.iterdir():
        if path.is_file() and path.suffix != '.lock':
            archive.write(path, arcname=path.name)
display(FileLink(str(zip_path)))